# Neighbourhood Comparison Membership Inference Attack Recreation

This notebook recreates the **neighbourhood comparison** membership inference attack summarized in `papers/summary/05_neighborhood.md`.

Primary source:

- Justus Mattern, Fatemehsadat Mireshghallah, Zhijing Jin, Bernhard Schölkopf, Mrinmaya Sachan, Taylor Berg-Kirkpatrick, *Membership Inference Attacks against Language Models via Neighbourhood Comparison*, Findings of the ACL 2023 / arXiv:2305.18462.
- Reference code repository (provided on request per the authors' ethics statement): https://github.com/mireshghallah/neighborhood-curvature-mia

The neighbourhood attack is a **reference-model-free** MIA. For a target text `x`, the attacker generates a set of near-identical *neighbour* texts via masked-LM (BERT) single-word substitutions, then compares `x`'s loss under the target model to the **mean** loss of its neighbours. Because the neighbours are practically interchangeable with `x` under any plausible text distribution, a target loss substantially below its neighbours' mean can only arise from overfitting — i.e., membership. Synthetic neighbours replace the reference model used in Likelihood Ratio Attacks (LiRA) as the difficulty-calibration signal, removing any need for in-domain reference data.

```
membership_score = mean_i L(neighbour_i) - L(x)     # higher => more likely member
```

The only heavyweight external dependencies (a GPT-2 target model and a BERT neighbour generator) are optional here: the runnable smoke test is pure-Python and imports nothing beyond the standard library.

## Baseline Attack Definition

**Threat model.** Grey-box access: the attacker can score candidate sequences under the target language model (compute per-token loss / log-perplexity) but has no model weights or gradients. Crucially, **no reference model and no access to the private training distribution are required** — the neighbours, generated by an off-the-shelf BERT with no domain adaptation, supply the difficulty calibration.

**Target record.** A candidate text sequence. Members are sequences present in the target model's fine-tuning set; non-members are distribution-matched held-out sequences. This is the **fine-tuning / multi-epoch MIA setting** (AG News summaries, Sentiment140 tweets, Wikitext-103 excerpts in the paper), not the WikiMIA pretraining setting.

**Neighbour generation (Algorithm 1 in the paper).** For text `x = (w^1 ... w^L)`, a masked LM (BERT) scores candidate replacement tokens at each position `i`. The suitability score normalises out the original token's own probability:

`p_swap(w_hat^i, w_tilde^i) = p_theta(w_tilde = w^i | x) / (1 - p_theta(w_hat = w^i | x))`.

Critically, the original token is **not** masked out; instead strong dropout (`p = 0.7`) is applied to the input embedding at position `i`, so the model still respects the original word's meaning when proposing replacements (preventing semantic flips such as "great" -> "bad"). Over all `m`-word swap combinations, the joint suitability is computed and the `n` highest-scoring neighbours are returned. Best configuration from the ablations: **`n = 100` neighbours, `m = 1` word replacement**, from a pretrained BERT (110M).

**Decision rule (Eq. 3).**

`A(x) = 1[ ( L(f_theta, x) - (1/n) sum_i L(f_theta, x_tilde_i) ) < gamma ]`

Membership is predicted when the target's loss minus the mean neighbour loss falls below `gamma`. For this notebook we report `membership_score = mean_neighbour_loss - target_loss`, chosen so a **higher** membership score means a more likely member (matching the `>=` threshold convention used across these recreations).

**Metrics.** TPR at low FPR (1%, 0.1%, 0.01%) and AUC. Paper headline (TPR@1%FPR): 8.29% (News) / 7.35% (Twitter) / 2.32% (Wiki); AUC 0.79 / 0.77 / 0.62 — beating LOSS and realistic LiRAs and competitive with the Oracle LiRA.

In [ ]:
from dataclasses import dataclass
from math import exp
from pathlib import Path
from typing import List, Sequence
from statistics import fmean

SOURCE_SUMMARY = Path("../../papers/summary/05_neighborhood.md")
ATTACK_NAME = "neighborhood"


def neighborhood_score(target_loss: float, neighbour_losses: Sequence[float]) -> float:
    """Core neighbourhood-comparison membership score.

    score = mean(neighbour_losses) - target_loss

    A memorized (member) record gets an unusually LOW loss from the target
    model relative to its interchangeable neighbours, so the mean neighbour
    loss exceeds the target loss and the score is large & positive. A
    non-member sits at roughly the same loss as its neighbours, so the score
    is near zero. Orientation: HIGHER => more likely member, matching the
    >= threshold convention used across these recreations. This is the
    negation of the paper's Eq. 3 quantity (L(x) - mean neighbour loss).
    """
    if not neighbour_losses:
        raise ValueError("Need at least one neighbour loss to calibrate.")
    return fmean(neighbour_losses) - target_loss


@dataclass(frozen=True)
class CandidateScore:
    text: str
    truth_member: bool
    target_loss: float               # mean per-token loss of x under the target model
    neighbour_losses: Sequence[float]  # per-neighbour mean per-token losses

    @property
    def mean_neighbour_loss(self) -> float:
        return fmean(self.neighbour_losses)

    @property
    def membership_score(self) -> float:
        # Higher => member (mean neighbour loss well above the target's loss).
        return neighborhood_score(self.target_loss, self.neighbour_losses)

    @property
    def target_perplexity(self) -> float:
        return exp(self.target_loss)

    @property
    def num_neighbours(self) -> int:
        return len(self.neighbour_losses)

## Optional Hugging Face Neighbour Generation and Scoring

These cells sketch a real recreation with a GPT-2 target and a BERT masked-LM neighbour generator. They are **not** executed in the smoke test and require `torch` + `transformers` plus model downloads.

`generate_neighbours_bert` implements the paper's Algorithm 1 for `m = 1`: for each token position it applies strong dropout to the *original* token embedding (rather than masking it), reads BERT's predictive distribution over the vocabulary, forms the normalised suitability score `p_swap`, and keeps the `n` highest-scoring single-token substitutions across all positions.

**Documented deviation.** A faithful implementation of the dropout-on-original-embedding trick requires a forward hook on BERT's word-embedding layer. The sketch below shows that faithful path; a simpler `[MASK]`-and-top-k fill is provided as `generate_neighbours_bert_maskfill` for environments where hooking the embeddings is inconvenient. The mask-fill variant can produce occasional semantic flips (the exact failure mode — e.g. "great" -> "bad" — that motivated the dropout trick), so it is documented as an approximation, not the paper's method.

In [ ]:
def generate_neighbours_bert(
    text: str,
    tokenizer_mlm,
    model_mlm,
    n: int = 100,
    dropout_p: float = 0.7,
    device: str = "cpu",
    max_length: int = 128,
):
    """Paper-faithful single-word (m=1) neighbour generation (Algorithm 1).

    For each token position i:
      1. Apply strong dropout (p=dropout_p) to the ORIGINAL token's input
         embedding (the token is NOT masked out) so BERT still respects the
         original word's meaning.
      2. Read BERT's distribution p_theta(v | x) at position i.
      3. Score each candidate replacement w_tilde by the normalised
         suitability p_swap = p(w_tilde) / (1 - p(original_token)).
    Collect all (position, replacement) candidates, rank by p_swap, and
    materialise the top-n as neighbour texts (each differs from x by one word).
    """
    import torch

    encoded = tokenizer_mlm(
        text, return_tensors="pt", truncation=True, max_length=max_length
    )
    input_ids = encoded["input_ids"].to(device)
    ids = input_ids[0]
    special = set(tokenizer_mlm.all_special_ids)

    embedding_layer = model_mlm.get_input_embeddings()
    candidates = []  # (p_swap, position, replacement_id)

    for pos in range(ids.shape[0]):
        original_id = int(ids[pos].item())
        if original_id in special:
            continue

        # Dropout on the ORIGINAL token embedding at this position only.
        base_embeds = embedding_layer(input_ids)  # (1, L, H)
        pos_embed = base_embeds[:, pos, :]
        dropped = torch.nn.functional.dropout(pos_embed, p=dropout_p, training=True)
        perturbed = base_embeds.clone()
        perturbed[:, pos, :] = dropped

        with torch.no_grad():
            logits = model_mlm(inputs_embeds=perturbed).logits
        probs = torch.softmax(logits[0, pos], dim=-1)
        p_original = float(probs[original_id].item())
        denom = max(1e-8, 1.0 - p_original)

        topk = torch.topk(probs, k=min(10, probs.shape[-1]))
        for score, cand_id in zip(topk.values.tolist(), topk.indices.tolist()):
            if cand_id == original_id or cand_id in special:
                continue
            p_swap = score / denom
            candidates.append((p_swap, pos, cand_id))

    candidates.sort(key=lambda c: c[0], reverse=True)
    neighbours = []
    for _, pos, cand_id in candidates[:n]:
        new_ids = ids.clone()
        new_ids[pos] = cand_id
        neighbours.append(
            tokenizer_mlm.decode(new_ids, skip_special_tokens=True)
        )
    return neighbours


def generate_neighbours_bert_maskfill(
    text: str,
    tokenizer_mlm,
    model_mlm,
    n: int = 100,
    device: str = "cpu",
    max_length: int = 128,
):
    """Simpler approximation: mask each position and take top-k fills.

    DEVIATION from the paper: masking the original token discards its meaning,
    so this can yield semantic flips the dropout trick avoids. Provided only
    for convenience; prefer generate_neighbours_bert for faithful results.
    """
    import torch

    encoded = tokenizer_mlm(
        text, return_tensors="pt", truncation=True, max_length=max_length
    )
    ids = encoded["input_ids"][0]
    special = set(tokenizer_mlm.all_special_ids)
    mask_id = tokenizer_mlm.mask_token_id

    candidates = []  # (prob, position, replacement_id)
    for pos in range(ids.shape[0]):
        original_id = int(ids[pos].item())
        if original_id in special:
            continue
        masked = ids.clone()
        masked[pos] = mask_id
        with torch.no_grad():
            logits = model_mlm(masked.unsqueeze(0).to(device)).logits
        probs = torch.softmax(logits[0, pos], dim=-1)
        topk = torch.topk(probs, k=min(10, probs.shape[-1]))
        for prob, cand_id in zip(topk.values.tolist(), topk.indices.tolist()):
            if cand_id == original_id or cand_id in special:
                continue
            candidates.append((prob, pos, cand_id))

    candidates.sort(key=lambda c: c[0], reverse=True)
    neighbours = []
    for _, pos, cand_id in candidates[:n]:
        new_ids = ids.clone()
        new_ids[pos] = cand_id
        neighbours.append(tokenizer_mlm.decode(new_ids, skip_special_tokens=True))
    return neighbours


def mean_token_nll_hf(model, tokenizer, text: str, device: str = "cpu", max_length: int = 256) -> float:
    """Return mean next-token loss (log-perplexity) for one text under a causal LM (e.g. GPT-2)."""
    import torch

    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    encoded = {key: value.to(device) for key, value in encoded.items()}
    input_ids = encoded["input_ids"]
    if input_ids.shape[-1] < 2:
        raise ValueError("Need at least two tokens to score a causal-LM sequence.")
    with torch.no_grad():
        outputs = model(**encoded, labels=input_ids)
    return float(outputs.loss.detach().cpu())


def score_texts_with_hf(
    target_model, target_tokenizer, tokenizer_mlm, model_mlm,
    texts: Sequence[str], labels: Sequence[bool],
    n: int = 100, device: str = "cpu", max_length: int = 256,
) -> List[CandidateScore]:
    """Full HF pipeline: for each candidate, generate n BERT neighbours and score all under the target."""
    rows = []
    for text, truth_member in zip(texts, labels):
        neighbours = generate_neighbours_bert(text, tokenizer_mlm, model_mlm, n=n, device=device, max_length=max_length)
        target_loss = mean_token_nll_hf(target_model, target_tokenizer, text, device=device, max_length=max_length)
        neighbour_losses = [
            mean_token_nll_hf(target_model, target_tokenizer, nb, device=device, max_length=max_length)
            for nb in neighbours
        ]
        rows.append(CandidateScore(text=text, truth_member=bool(truth_member),
                                   target_loss=target_loss, neighbour_losses=neighbour_losses))
    return rows

## Thresholding and Metrics

The paper ranks samples by the calibrated score and reports TPR at low FPR and AUC. For small controlled trials, this notebook additionally reports thresholded confusion counts, TPR, TNR, attack advantage, accuracy, precision, recall, and F1, plus a threshold-free ROC-AUC (the Mann–Whitney form copied from the zlib adaptation).

In [ ]:
def predict_membership(rows: Sequence[CandidateScore], threshold: float) -> List[bool]:
    return [row.membership_score >= threshold for row in rows]


def confusion_counts(labels: Sequence[bool], preds: Sequence[bool]):
    tp = sum(1 for y, p in zip(labels, preds) if y and p)
    tn = sum(1 for y, p in zip(labels, preds) if not y and not p)
    fp = sum(1 for y, p in zip(labels, preds) if not y and p)
    fn = sum(1 for y, p in zip(labels, preds) if y and not p)
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn}


def roc_auc(labels: Sequence[bool], scores: Sequence[float]) -> float:
    """Rank-based ROC-AUC (probability a random member outranks a random non-member)."""
    pos = [s for y, s in zip(labels, scores) if y]
    neg = [s for y, s in zip(labels, scores) if not y]
    if not pos or not neg:
        return float("nan")
    wins = 0.0
    for p in pos:
        for n in neg:
            wins += 1.0 if p > n else (0.5 if p == n else 0.0)
    return wins / (len(pos) * len(neg))


def metric_summary(rows: Sequence[CandidateScore], preds: Sequence[bool]):
    labels = [row.truth_member for row in rows]
    counts = confusion_counts(labels, preds)
    tp, tn, fp, fn = counts["tp"], counts["tn"], counts["fp"], counts["fn"]
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tpr
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        **counts,
        "tpr": tpr,
        "tnr": tnr,
        "adv": 0.5 * tpr + 0.5 * tnr,
        "accuracy": (tp + tn) / len(labels) if labels else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc(labels, [row.membership_score for row in rows]),
    }


def percentile_threshold(rows: Sequence[CandidateScore], member_fraction: float = 0.5) -> float:
    scores = sorted(row.membership_score for row in rows)
    if not scores:
        raise ValueError("Cannot threshold an empty score list.")
    index = max(0, min(len(scores) - 1, int((1.0 - member_fraction) * len(scores))))
    return scores[index]

## Synthetic Smoke Recreation

The synthetic table emulates the expected neighbourhood signal without any model:

- **Members** are records the target model has memorized: their target loss sits **well below** the mean loss of their interchangeable neighbours (the model overfit the exact string but not its one-word variants). After `mean_neighbour_loss - target_loss` this yields a **large positive** membership score.
- **Non-members** are ordinary held-out text: the model has no special affinity for the exact string over its neighbours, so the target loss is **roughly equal** to the neighbour mean and the score is near zero.

Because the members' target loss is depressed relative to neighbours while non-members' is not, members outrank non-members and the ranking (AUC) is perfect. This is a correctness check on the scoring, calibration, and metrics pipeline — not a substitute for the full GPT-2 / BERT experiment.

In [ ]:
def synthetic_neighborhood_scores() -> List[CandidateScore]:
    # Members: target loss WELL BELOW the neighbour mean (overfit the exact string).
    # Non-members: target loss ~ neighbour mean (no special memorization).
    return [
        CandidateScore(
            "Patient Ana Ortiz, MRN 84213, was prescribed 12 units of insulin nightly.",
            True, target_loss=0.85,
            neighbour_losses=[2.30, 2.45, 2.20, 2.55, 2.35, 2.40, 2.25, 2.50],
        ),
        CandidateScore(
            "API_SECRET_KEY = sk-live-9f3a2b7c4d8e1f6a0c5b2d9e7f4a1c3b",
            True, target_loss=0.60,
            neighbour_losses=[2.10, 2.05, 2.20, 1.95, 2.15, 2.00, 2.25, 2.10],
        ),
        CandidateScore(
            "The committee will reconvene next quarter to review the proposal.",
            False, target_loss=2.42,
            neighbour_losses=[2.40, 2.38, 2.45, 2.41, 2.39, 2.44, 2.37, 2.43],
        ),
        CandidateScore(
            "Public clinic reminder: bring your insurance card and arrive early.",
            False, target_loss=2.28,
            neighbour_losses=[2.25, 2.31, 2.27, 2.30, 2.26, 2.29, 2.24, 2.32],
        ),
    ]


def run_recreation_smoke_test():
    rows = synthetic_neighborhood_scores()
    threshold = percentile_threshold(rows, member_fraction=0.5)
    preds = predict_membership(rows, threshold=threshold)
    metrics = metric_summary(rows, preds)

    # The two memorized records must rank above both held-out records.
    assert metrics["tp"] == 2, metrics
    assert metrics["tn"] == 2, metrics
    assert metrics["adv"] == 1.0, metrics
    assert metrics["roc_auc"] == 1.0, metrics

    # Neighbourhood calibration sanity: members' target loss sits far below
    # their neighbour mean; non-members' target loss ~ neighbour mean.
    members = [r for r in rows if r.truth_member]
    non_members = [r for r in rows if not r.truth_member]
    assert all(m.membership_score > 1.0 for m in members), "members should have a large positive gap"
    assert all(abs(nm.membership_score) < 0.2 for nm in non_members), "non-members should sit near zero"
    assert min(m.membership_score for m in members) > max(nm.membership_score for nm in non_members), \
        "members must outrank non-members"

    return {
        "threshold": threshold,
        "metrics": metrics,
        "ranking": [
            {"text": r.text[:40], "member": r.truth_member,
             "target_loss": round(r.target_loss, 3),
             "mean_neighbour_loss": round(r.mean_neighbour_loss, 3),
             "n_neighbours": r.num_neighbours,
             "score": round(r.membership_score, 6)}
            for r in sorted(rows, key=lambda r: r.membership_score, reverse=True)
        ],
    }

smoke_result = run_recreation_smoke_test()
smoke_result

## How to Run a Real Recreation

1. Load the target causal LM with `AutoModelForCausalLM` (the paper fine-tunes GPT-2 base, 117M) plus its tokenizer. Any fine-tuned checkpoint works.
2. Load an off-the-shelf `bert-base-uncased` masked LM + tokenizer as the neighbour generator (no domain adaptation — the paper uses the pretrained 110M BERT).
3. Collect matched candidate member and non-member texts (the paper splits AG News / Sentiment140 / Wikitext-103 into disjoint train/non-train halves).
4. Call `score_texts_with_hf(target_model, target_tokenizer, bert_tokenizer, bert_model, texts, labels, n=100)`. This generates `n = 100` single-word (`m = 1`) neighbours per candidate via the dropout-on-embedding trick, scores the candidate and all neighbours under the target, and computes `membership_score = mean_neighbour_loss - target_loss`. **No reference model is needed** — the neighbours are the calibration.
5. Rank by `membership_score`; report `roc_auc` from `metric_summary` and TPR at low FPR (1%, 0.1%, 0.01%) by sweeping the threshold.

For the federated-learning fine-tuning adaptation of this attack, see `../adaptations/neighborhood_adaptations.ipynb`.